# Phase 2 — Bulk training-data generation on Kaggle (vLLM + self-judge)

This notebook generates the **training** set for the Vietnamese in-car dialogue
*rewrite/retrieval* task, following `src/data/gen_data.md`.

**Pipeline position**
- **Phase 1 (done):** `build_benchmark.py` → GPT-4o made the *hard eval* set
  `data/bench/dialogues_bench.jsonl` (8 patterns). We reuse it here as the
  **seed / few-shot pool**.
- **Phase 2 (this notebook):** a quantized **Qwen2.5-14B-Instruct-AWQ** served by
  **vLLM** bulk-generates dialogues, and the **same model self-judges** them
  (different system prompt, low temperature). Output is the Llama-Factory
  `conversations` schema — drop-in for your existing training config.
- **Phase 3:** train the 1.5B LoRA student on the merged data.

**Spec mapping**
- Multi-turn, **1–4 user / 0–3 bot** turns, explicit **turn-count × domain** quotas.
- **≈2/3 context-required, ≈1/3 no-context** — teaches *when to retrieve vs keep-as-is*.
- **No-repeat-action**: never re-issue an action the bot already completed.
- **Keep-as-is**: if the final user turn is already complete, the rewrite stays faithful to it.
- High temperature to **generate**, temperature 0 to **judge**, two distinct system prompts.
- vLLM continuous batching = the "multithreading" the spec asks for (one big
  `.generate()` saturates the GPU — no sequential calls).

> **Settings:** GPU = **T4 ×2**, Internet = **ON**. Add the repo (or just
> `data/bench/dialogues_bench.jsonl`) as a Kaggle dataset input, or clone the repo below.

## 1. Install

In [ ]:
# vLLM pulls a matching torch/transformers. Pin a known-good vLLM for T4 (sm75, AWQ).
!pip -q install "vllm==0.6.3.post1" "transformers>=4.45" 2>/dev/null
import torch, vllm
print("torch", torch.__version__, "| cuda", torch.cuda.is_available(),
      "| gpus", torch.cuda.device_count(), "| vllm", vllm.__version__)

## 2. Config

Everything is parameterised here. Bump `TARGET_TOTAL` for bigger runs — the wave
loop + on-disk cache make the notebook resumable across Kaggle sessions.

In [ ]:
import os, glob

# --- teacher model (trained-quantized AWQ, per gen_data.md) ---
MODEL_ID         = "Qwen/Qwen2.5-14B-Instruct-AWQ"
TENSOR_PARALLEL  = 2          # 2x T4
MAX_MODEL_LEN    = 4096
GPU_MEM_UTIL     = 0.92

# --- volume + mix ---
TARGET_TOTAL     = 3000
CONTEXT_FRACTION = 2 / 3      # 2/3 need context, 1/3 don't (spec)

# --- generation (high temp = diverse) ---
GEN_TEMPERATURE  = 1.0
GEN_TOP_P        = 0.95
GEN_MAX_TOKENS   = 3072
SAMPLES_PER_CALL = 5          # samples requested per generation prompt
N_FEWSHOT        = 3          # seed examples injected per prompt

# --- judge (temp 0 = deterministic) ---
JUDGE_TEMPERATURE = 0.0
JUDGE_MAX_TOKENS  = 192

# --- wave loop ---
MAX_WAVES        = 14
OVERSAMPLE       = 2.2        # validators + judge reject ~40-55%
SEED             = 42

# --- paths: find the Phase-1 bench in a Kaggle dataset OR the repo ---
def _find_bench():
    cands  = glob.glob("/kaggle/input/**/dialogues_bench.jsonl", recursive=True)
    cands += ["data/bench/dialogues_bench.jsonl",
              "/kaggle/working/qwen-lora-finetuning/data/bench/dialogues_bench.jsonl"]
    for c in cands:
        if os.path.exists(c):
            return c
    return None

BENCH_PATH = _find_bench()
OUT_DIR    = "/kaggle/working/data/train"
os.makedirs(OUT_DIR, exist_ok=True)
OUT_PATH   = os.path.join(OUT_DIR, "dialogues_train_vllm.jsonl")
CACHE_PATH = os.path.join(OUT_DIR, ".cache_gen_vllm.jsonl")

print("bench :", BENCH_PATH or "NOT FOUND — few-shot will be skipped (lower quality)")
print("out   :", OUT_PATH)
print("cache :", CACHE_PATH)

## 3. System prompts

Three strings: the **training** system prompt (must match `src/data/prompts.py`
so generated data aligns with deployment), the **generator** prompt (diverse), and
a **very explicit judge** prompt (deterministic scoring).

In [ ]:
# Must stay identical to src/data/prompts.py SYSTEM_PROMPT_FOR_TRAINING.
SYSTEM_PROMPT_FOR_TRAINING = """Bạn là một module xử lý NGÔN NGỮ cho hệ thống trợ lý trong xe.

Khi người dùng gửi yêu cầu có tag <REWRITE>, bạn PHẢI:
1. Viết lại câu ở phía sau tag này thành MỘT câu hoàn chỉnh, đầy đủ ý nghĩa.
2. Ngắn gọn, rõ nghĩa.
3. Chỉ sử dụng thông tin có trong hội thoại trước đó nếu cần — KHÔNG thêm thông tin mới.
4. Chỉ trả về JSON hợp lệ dạng: {"rewrite_message": "..."}"""

REWRITE_TAG = "<REWRITE>"


GEN_SYSTEM = """Bạn là chuyên gia tạo dữ liệu HUẤN LUYỆN tiếng Việt cho task rewrite hội thoại trong xe ô tô.

Mỗi mẫu gồm một đoạn hội thoại nhiều lượt (xen kẽ user/bot, kết thúc bằng user) và MỘT câu rewrite chuẩn cho lượt user CUỐI.

NGUYÊN TẮC SINH:
- Đa dạng domain; slot phải CỤ THỂ và có thật (tên người, địa điểm Hà Nội/Sài Gòn thật, bài Vpop thật, hãng EV thật, nhiệt độ/khoảng cách hợp lý).
- Văn nói tự nhiên trong xe: có thể lược chủ ngữ, nói nhanh, slang nhẹ.
- Biến tấu opener của câu rewrite — KHÔNG luôn mở đầu bằng "Tôi muốn...". Dùng "Hãy...", "Làm ơn...", "Cho tôi...", "Mình cần...", câu mệnh lệnh ngắn.
- KHÔNG lặp scenario/từ vựng giữa các mẫu trong cùng batch.

HAI LOẠI MẪU (rất quan trọng):
(A) context_required = true — lượt user CUỐI THIẾU thông tin (đại từ "cái đó/bài đó", quá ngắn, slot rải rác ở các lượt trước, hoặc tham chiếu ngầm). Câu rewrite PHẢI giải/gộp slot từ các lượt TRƯỚC.
    • TUYỆT ĐỐI KHÔNG lặp lại hành động mà BOT ĐÃ LÀM XONG ở lượt trước — chỉ rewrite hành động CHƯA được thực hiện.
(B) context_required = false — lượt user CUỐI ĐÃ ĐẦY ĐỦ, tự nó thực hiện được (vd "Bật đèn pha như tôi yêu cầu đi"). Câu rewrite GIỮ NGUYÊN ý lượt cuối, chỉ làm sạch/chuẩn hoá; KHÔNG kéo slot lạ từ các lượt trước.

Giữ negation ("đừng/trừ/không/tránh") và compound ("và/rồi/sau đó") nếu có — KHÔNG làm mất.

Chỉ trả về JSON đúng schema yêu cầu, không thêm chữ nào ngoài JSON."""


JUDGE_SYSTEM = """Bạn là GIÁM KHẢO chấm câu rewrite tiếng Việt cho task hội thoại trợ lý xe. Hãy chấm CỰC KỲ NGHIÊM KHẮC và theo đúng luật.

Bạn nhận:
- dialogue: lịch sử hội thoại (xen kẽ user/bot, kết thúc bằng user).
- context_required: true nếu lượt user cuối cần ngữ cảnh để hiểu, false nếu lượt cuối đã đầy đủ.
- prediction: câu rewrite cần chấm.

CHO score = 1 KHI VÀ CHỈ KHI prediction thoả MỌI điều:
1. Là MỘT câu tiếng Việt hoàn chỉnh, đứng một mình hiểu được, đúng ý định lượt user cuối.
2. Bảo toàn TẤT CẢ slot cần thiết (tên người, địa điểm, số, nhiệt độ, bài hát, hãng, chế độ).
3. KHÔNG thêm thông tin/slot bịa, KHÔNG kéo nhầm số hay slot mà bot chỉ nhắc tới nhưng user không yêu cầu.
4. Nếu context_required=true: phải GIẢI đại từ/tham chiếu và GỘP đủ slot rải rác từ các lượt trước; và KHÔNG lặp lại hành động bot đã làm xong.
5. Nếu context_required=false: phải bám sát lượt user cuối, KHÔNG thêm slot từ các lượt trước.
6. Giữ đúng polarity negation (đừng/trừ/không/tránh) và đủ cả 2 vế của compound intent.

NGƯỢC LẠI cho score = 0 (thiếu slot, sai intent, bịa thông tin, mất negation, lặp hành động đã làm, hoặc câu không tự đứng được).

Chỉ trả về JSON: {"score": 0 hoặc 1, "reason": "1 câu ngắn tiếng Việt"}."""

## 4. Quota plan (turn-count × context × domain)

Explicit ratios per the spec. `context_required=True` skews to longer dialogues
(slots to retrieve); `False` skews to short/single turns (already complete).

In [ ]:
import random
random.seed(SEED)

DOMAIN_WEIGHTS = {
    "navigation": 1.0, "climate": 1.0, "music": 1.0, "calling": 1.0,
    "messaging": 2.0, "charging": 2.5, "smart_home": 3.0,
    "vehicle": 3.0, "driver_assist": 3.0,
}
DOMAINS = list(DOMAIN_WEIGHTS)

# user_turns distribution within each context class.
TURN_DIST = {
    True:  {2: 0.45, 3: 0.35, 4: 0.20},   # context-required → ≥2 user turns
    False: {1: 0.55, 2: 0.45},            # no-context → mostly single turn
}

def _alloc(total, dist):
    out = {k: round(total * w) for k, w in dist.items()}
    out[max(out)] += total - sum(out.values())  # fix rounding
    return out

n_ctx   = round(TARGET_TOTAL * CONTEXT_FRACTION)
n_noctx = TARGET_TOTAL - n_ctx

QUOTA = {}  # (context_required, user_turns) -> target count
for ut, c in _alloc(n_ctx,   TURN_DIST[True]).items():  QUOTA[(True,  ut)] = c
for ut, c in _alloc(n_noctx, TURN_DIST[False]).items(): QUOTA[(False, ut)] = c

print(f"context-required: {n_ctx}  |  no-context: {n_noctx}")
for k in sorted(QUOTA): print(f"  ctx={str(k[0]):5s} user_turns={k[1]} -> {QUOTA[k]}")

def pick_domains(k=3):
    return random.choices(DOMAINS, weights=[DOMAIN_WEIGHTS[d] for d in DOMAINS], k=k)

## 5. Seed few-shot pool (from the Phase-1 bench)

In [ ]:
import json

def bench_to_compact(rec):
    convs = [c for c in rec["conversations"] if c["from"] != "system"]
    gold  = json.loads(convs[-1]["value"])["rewrite_message"]
    turns = []
    for c in convs[:-1]:
        role = "user" if c["from"] == "human" else "bot"
        v = c["value"]
        if v.startswith("<REWRITE>\n"):
            v = v[len("<REWRITE>\n"):]
        turns.append({"role": role, "content": v})
    return {"turns": turns, "rewrite": gold,
            "domain": rec.get("meta", {}).get("domain", "navigation")}

SEED_POOL = []
if BENCH_PATH:
    for line in open(BENCH_PATH, encoding="utf-8"):
        try:
            SEED_POOL.append(bench_to_compact(json.loads(line)))
        except Exception:
            pass
print(f"seed pool: {len(SEED_POOL)} examples")

def fewshot_block(k=N_FEWSHOT):
    if not SEED_POOL:
        return ""
    picks = random.sample(SEED_POOL, min(k, len(SEED_POOL)))
    ex = [{"turns": p["turns"], "rewrite": p["rewrite"], "domain": p["domain"]} for p in picks]
    return ("Vài ví dụ THAM KHẢO về độ khó & văn phong (đừng sao chép):\n"
            + json.dumps(ex, ensure_ascii=False, indent=1) + "\n\n")

## 6. Load the vLLM engine

AWQ on T4 (sm75) → `quantization="awq"`, `dtype="float16"` (T4 has no bf16).
Loading takes a few minutes; run once per session.

In [ ]:
from vllm import LLM, SamplingParams
from transformers import AutoTokenizer

tok = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
llm = LLM(
    model=MODEL_ID,
    quantization="awq",
    tensor_parallel_size=TENSOR_PARALLEL,
    max_model_len=MAX_MODEL_LEN,
    gpu_memory_utilization=GPU_MEM_UTIL,
    dtype="float16",
    trust_remote_code=True,
    enforce_eager=False,
)

def build_prompt(system, user):
    return tok.apply_chat_template(
        [{"role": "system", "content": system}, {"role": "user", "content": user}],
        tokenize=False, add_generation_prompt=True,
    )

GEN_SP   = SamplingParams(temperature=GEN_TEMPERATURE, top_p=GEN_TOP_P, max_tokens=GEN_MAX_TOKENS)
JUDGE_SP = SamplingParams(temperature=JUDGE_TEMPERATURE, max_tokens=JUDGE_MAX_TOKENS)
print("engine ready")

## 7. Validators + Llama-Factory conversion

Cheap heuristic gate before the (expensive) self-judge: structure, turn count,
no hallucinated numbers, and the spec's keep-as-is / context rules.

In [ ]:
import re, unicodedata

NUMBER_RE = re.compile(r"\d+(?:[.,]\d+)?")

def _norm(t):
    t = unicodedata.normalize("NFC", t).lower().strip()
    return re.sub(r"[\s,.!?;:\"'()\-]+", " ", t).strip()

def _nums(t):
    return {m.group(0).replace(",", ".") for m in NUMBER_RE.finditer(t)}

REJECT = {}
def _rej(reason):
    REJECT[reason] = REJECT.get(reason, 0) + 1
    return None

def validate(s, ctx_req, want_ut):
    if not isinstance(s, dict):                       return _rej("not_dict")
    turns, rw = s.get("turns"), (s.get("rewrite") or "").strip()
    domain = (s.get("domain") or "").strip()
    if not isinstance(turns, list) or not rw or not domain: return _rej("missing_field")
    if not (5 <= len(rw) <= 240):                     return _rej("rewrite_length")

    cleaned, prev = [], None
    for t in turns:
        if not isinstance(t, dict):                   return _rej("turn_not_dict")
        role, content = t.get("role"), (t.get("content") or "").strip()
        if role not in ("user", "bot") or not content: return _rej("bad_turn")
        if role == prev:                              return _rej("not_alternating")
        prev = role
        cleaned.append({"role": role, "content": content})
    if not cleaned or cleaned[0]["role"] != "user":   return _rej("not_start_user")
    if cleaned[-1]["role"] != "user":                 return _rej("not_end_user")

    n_user = sum(1 for t in cleaned if t["role"] == "user")
    if n_user != want_ut:                             return _rej("wrong_user_turns")

    last_user = cleaned[-1]["content"]
    all_nums  = _nums(" ".join(t["content"] for t in cleaned))
    rw_nums   = _nums(rw)
    if not rw_nums.issubset(all_nums):                return _rej("hallucinated_number")

    if ctx_req:
        if _norm(last_user) == _norm(rw):             return _rej("ctx_but_identical")
    else:
        # keep-as-is: rewrite must not pull numbers absent from the final user turn
        if not rw_nums.issubset(_nums(last_user)):    return _rej("noctx_pulled_number")

    return {"turns": cleaned, "rewrite": rw, "domain": domain,
            "context_required": ctx_req, "user_turns": n_user,
            "rationale": (s.get("rationale") or "").strip()}

def to_lf(s):
    convs = [{"from": "system", "value": SYSTEM_PROMPT_FOR_TRAINING}]
    for t in s["turns"][:-1]:
        convs.append({"from": "human" if t["role"] == "user" else "gpt", "value": t["content"]})
    convs.append({"from": "human", "value": f"{REWRITE_TAG}\n{s['turns'][-1]['content']}"})
    convs.append({"from": "gpt", "value": json.dumps({"rewrite_message": s["rewrite"]}, ensure_ascii=False)})
    return {"conversations": convs,
            "meta": {"domain": s["domain"], "context_required": s["context_required"],
                     "user_turns": s["user_turns"], "total_turns": len(s["turns"]),
                     "source": "qwen2.5-14b-awq"}}

## 8. Generation + self-judge functions

`json.loads` first, regex-extract fallback for stray prose. The judge runs the
**same model** with the strict prompt at temperature 0.

In [ ]:
def schema_instructions(want_ut, ctx_req):
    bots = want_ut - 1
    kind = ("context_required=true (lượt user cuối THIẾU thông tin, rewrite phải dùng ngữ cảnh trước)"
            if ctx_req else
            "context_required=false (lượt user cuối ĐÃ đầy đủ, rewrite giữ nguyên ý lượt cuối)")
    return (
        f"Sinh {SAMPLES_PER_CALL} mẫu KHÁC NHAU, loại: {kind}.\n"
        f"Mỗi hội thoại có ĐÚNG {want_ut} lượt user và {bots} lượt bot, "
        f"xen kẽ user-bot-...-user, KẾT THÚC bằng user.\n\n"
        "Schema JSON bắt buộc:\n"
        '{ "samples": [ { "turns": [ {"role":"user","content":"..."}, {"role":"bot","content":"..."} ], '
        '"rewrite":"câu rewrite cho lượt user cuối", "domain":"navigation|climate|music|calling|messaging|'
        'charging|smart_home|vehicle|driver_assist", "rationale":"1 câu vì sao" } ] }'
    )

_JSON_RE = re.compile(r"\{.*\}", re.DOTALL)

def parse_samples(text):
    try:
        return json.loads(text).get("samples", [])
    except Exception:
        m = _JSON_RE.search(text)
        if not m:
            return []
        try:
            return json.loads(m.group(0)).get("samples", [])
        except Exception:
            return []

def parse_score(text):
    try:
        return int(json.loads(text).get("score", 0))
    except Exception:
        m = _JSON_RE.search(text)
        if m:
            try:
                return int(json.loads(m.group(0)).get("score", 0))
            except Exception:
                pass
        return 1 if re.search(r'"score"\s*:\s*1', text) else 0

def dialogue_str(s):
    return "\n".join(f"{t['role']}: {t['content']}" for t in s["turns"])

def judge_prompt(s):
    return build_prompt(JUDGE_SYSTEM,
        f"dialogue:\n{dialogue_str(s)}\n\n"
        f"context_required: {str(s['context_required']).lower()}\n\n"
        f"prediction:\n{s['rewrite']}\n\nChấm theo schema.")

## 9. Main wave loop

Each wave: build deficit-targeted prompts → one batched `.generate()` (high temp)
→ validate → one batched judge `.generate()` (temp 0) → keep `score==1`, dedup,
append to cache. Repeat until quotas fill or `MAX_WAVES`.

In [ ]:
import math
from collections import Counter

# resume from cache
accepted = {}                      # (final_user, rewrite) -> sample
def _key(s): return (s["turns"][-1]["content"], s["rewrite"])
if os.path.exists(CACHE_PATH):
    for line in open(CACHE_PATH, encoding="utf-8"):
        s = json.loads(line); accepted[_key(s)] = s
    print(f"resumed {len(accepted)} cached samples")

cache_fh = open(CACHE_PATH, "a", encoding="utf-8")

def have_counts():
    c = Counter((s["context_required"], s["user_turns"]) for s in accepted.values())
    return c

for wave in range(1, MAX_WAVES + 1):
    have = have_counts()
    deficits = {b: QUOTA[b] - have.get(b, 0) for b in QUOTA if QUOTA[b] - have.get(b, 0) > 0}
    if not deficits:
        print("all quotas filled ✓"); break

    # build prompts for this wave
    prompts, specs = [], []
    for (ctx, ut), need in deficits.items():
        n_calls = max(1, math.ceil(need * OVERSAMPLE / SAMPLES_PER_CALL))
        for _ in range(n_calls):
            doms = pick_domains(3)
            user = (fewshot_block()
                    + schema_instructions(ut, ctx)
                    + f"\n\nƯu tiên các domain: {', '.join(doms)}. seed={random.randint(1, 10_000_000)}")
            prompts.append(build_prompt(GEN_SYSTEM, user))
            specs.append((ctx, ut))

    print(f"\n── wave {wave}: deficits={deficits} → {len(prompts)} gen prompts")
    outs = llm.generate(prompts, GEN_SP)

    # validate
    cands = []
    for (ctx, ut), o in zip(specs, outs):
        for raw in parse_samples(o.outputs[0].text):
            v = validate(raw, ctx, ut)
            if v and _key(v) not in accepted:
                cands.append(v)
    print(f"   validated candidates: {len(cands)}")
    if not cands:
        continue

    # self-judge (temp 0)
    jouts = llm.generate([judge_prompt(s) for s in cands], JUDGE_SP)
    kept = 0
    for s, jo in zip(cands, jouts):
        if parse_score(jo.outputs[0].text) == 1:
            k = _key(s)
            if k in accepted:
                continue
            accepted[k] = s
            cache_fh.write(json.dumps(s, ensure_ascii=False) + "\n"); cache_fh.flush()
            kept += 1
    print(f"   judged → kept {kept} (total {len(accepted)}/{TARGET_TOTAL})")

cache_fh.close()
print(f"\nDONE — {len(accepted)} accepted samples")

## 10. Convert → Llama-Factory, save & report

In [ ]:
records, seen = [], set()
for s in accepted.values():
    k = _key(s)
    if k in seen:
        continue
    seen.add(k)
    records.append(to_lf(s))

with open(OUT_PATH, "w", encoding="utf-8") as f:
    for r in records:
        f.write(json.dumps(r, ensure_ascii=False) + "\n")
print(f"wrote {len(records)} samples → {OUT_PATH}\n")

from collections import Counter
by_ctx = Counter(r["meta"]["context_required"] for r in records)
by_ut  = Counter(r["meta"]["user_turns"] for r in records)
by_dom = Counter(r["meta"]["domain"] for r in records)
print("context_required:", dict(by_ctx),
      f"  (target {CONTEXT_FRACTION:.0%} True)")
print("user_turns      :", dict(sorted(by_ut.items())))
print("domain          :")
for d, n in by_dom.most_common():
    print(f"   {d:14s} {n}")
if REJECT:
    print("\nvalidator rejections:")
    for k, v in sorted(REJECT.items(), key=lambda x: -x[1]):
        print(f"   {k:24s} {v}")

print("\n--- sample preview ---")
for r in records[:2]:
    print(json.dumps(r, ensure_ascii=False, indent=2))

## 11. Next steps

- **Augment** (optional): run `src/data/paraphrase_with_llm.py` on `OUT_PATH` for
  +1 paraphrase per sample with slot preservation.
- **Merge** with any existing template/legacy train data, shuffle, then split.
- **Register** the file in Llama-Factory `dataset_info.json` and point your
  `configs/qwen_lora_sft.yaml` at it for Phase-3 training.
- **Evaluate** the trained adapter on the hard bench via `src/eval/eval_bench.py`.

Download `dialogues_train_vllm.jsonl` from the Kaggle output, or push it back to
your repo/HF dataset.

In [ ]:
# quick merge + shuffle helper (optional)
import json, random
EXTRA = []  # e.g. ["/kaggle/input/.../dialogues_train_template.jsonl"]
merged, seen = [], set()
for p in [OUT_PATH, *EXTRA]:
    if not os.path.exists(p):
        continue
    for line in open(p, encoding="utf-8"):
        r = json.loads(line)
        last_h = next(c["value"] for c in reversed(r["conversations"]) if c["from"] == "human")
        k = (last_h, r["conversations"][-1]["value"])
        if k in seen:
            continue
        seen.add(k); merged.append(r)
random.shuffle(merged)
mp = os.path.join(OUT_DIR, "dialogues_train_merged.jsonl")
with open(mp, "w", encoding="utf-8") as f:
    for r in merged:
        f.write(json.dumps(r, ensure_ascii=False) + "\n")
print(f"merged {len(merged)} → {mp}")